# Weight Importance Analysis

Identify which weights matter most via Fisher information, magnitude pruning,
lottery tickets, and redundancy analysis. Essential for understanding which
parameters the model actually uses.

This notebook covers the `irtk.weight_importance` module.

## Why This Matters

Weight importance analysis identifies which parameters matter most for model behavior. Fisher information, magnitude pruning, and lottery ticket analysis reveal the sparse substructure within dense networks — often a small fraction of weights carry most of the model's capability.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import weight_importance

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 1. Fisher Information Importance

Which parameters most affect the loss?

In [ ]:
prompts = ["The Eiffel Tower is located in", "The capital of France is"]
token_seqs = [model.to_tokens(p) for p in prompts]

result = weight_importance.fisher_information_importance(
    model, token_seqs, "blocks.6.mlp.W_in"
)

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(np.log10(result['importance'] + 1e-10), cmap='hot', aspect='auto')
ax.set_xlabel('d_mlp dimension')
ax.set_ylabel('d_model dimension')
ax.set_title('Fisher Information: MLP Layer 6 W_in (log scale)')
plt.colorbar(im, ax=ax, label='log10(Fisher info)')
plt.tight_layout()
plt.show()

print(f"Mean importance: {result['mean_importance']:.6f}")
print(f"Top 90% params in {result['top_fraction_90']:.1%} of parameters")

## 2. Lottery Ticket Mask

Find the minimal subset of weights that maintains performance.

In [ ]:
result = weight_importance.lottery_ticket_mask(
    model, token_seqs, "blocks.6.mlp.W_in",
    target_sparsity=0.5, method="magnitude"
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.imshow(result['mask'].astype(float), cmap='gray', aspect='auto')
ax.set_title(f'Lottery Ticket Mask (keeping {result["n_kept"]}/{result["n_total"]} params)')
ax.set_xlabel('d_mlp dimension')
ax.set_ylabel('d_model dimension')
plt.tight_layout()
plt.show()

print(f"Sparsity: {result['actual_sparsity']:.2%}")

## 3. Magnitude Pruning Curve

How does performance degrade as we prune more weights?

In [ ]:
paris_id = tokenizer.encode(" Paris")[0]
def metric(logits):
    return float(logits[-1, paris_id])

result = weight_importance.magnitude_pruning_curve(
    model, token_seqs, "blocks.6.mlp.W_in", metric
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(result['sparsity_levels'], result['metrics'], 'bo-')
if result['critical_sparsity'] is not None:
    ax.axvline(result['critical_sparsity'], color='red', linestyle='--',
               label=f'Critical sparsity = {result["critical_sparsity"]:.1f}')
ax.set_xlabel('Sparsity (fraction pruned)')
ax.set_ylabel('Metric (Paris logit)')
ax.set_title('Pruning Curve: MLP Layer 6')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Parameter Redundancy

Which weight rows are redundant (cosine-similar)?

In [ ]:
paths = [f"blocks.{i}.mlp.W_in" for i in range(model.cfg.n_layers)]
result = weight_importance.parameter_redundancy_analysis(model, paths)

fig, ax = plt.subplots(figsize=(10, 4))
scores = [result['redundancy_scores'][p] for p in paths]
ax.bar(range(len(paths)), scores, color='steelblue')
ax.set_xticks(range(len(paths)))
ax.set_xticklabels([f'Layer {i}' for i in range(len(paths))], rotation=45, ha='right')
ax.set_ylabel('Redundancy Score')
ax.set_title('MLP W_in Row Redundancy by Layer')
plt.tight_layout()
plt.show()

print(f"Most redundant: {result['most_redundant']}")

## Summary

| Function | Purpose |
|----------|--------|
| `fisher_information_importance()` | Weight importance via Fisher information |
| `activation_variance_importance()` | Importance from weight magnitude |
| `lottery_ticket_mask()` | Minimal weight subset for performance |
| `magnitude_pruning_curve()` | Performance vs sparsity curve |
| `parameter_redundancy_analysis()` | Find redundant weight rows |